# Feature Extraction with MEGaNorm

In this notebook, we demonstrate how to use **MEGaNorm** to extract features from resting-state Magnetoencephalography (MEG) and Electroencephalography (EEG) recordings in parallel. The package automatically locates the M/EEG recordings and performs the following processing steps:

* Load the recordings
* preproccess the recordings
* Perform source localization (MEG only)
* Apply spectral parameterization
* Extract features

> **Note 2:** MEGaNorm is currently under active development. Additional features and improvements are planned, and some components are still being refined. If you are interested in contributing or collaborating, we would be happy to discuss potential collaborations.

## Notebook Overview

This notebook is organized as follows:

1. Setting paths
2. Importing the required packages
3. Defining the processing configuration
4. Specifying participant demographics
5. Extracting features


## Setting Paths

To enable **MEGaNorm** to locate your data, you must define a Python dictionary named `datasets` and pass it to the package. This dictionary gives necessary cues to MEGaNorm to find the M/EEG recordings, empty-room recordings, event files, and FreeSurfer derivatives.

This flexible approach allows MEGaNorm to work with datasets that are **not fully BIDS-compliant**, provided that the required files follow a consistent naming convention. **FreeSurfer outputs, however, must follow the standard FreeSurfer directory structure.** Therefore, please pay close attention to the paths and filename conventions described below.


### `datasets`

`datasets` is a nested Python dictionary in which each key corresponds to a dataset name (e.g., `'UK_biobank'`, `'CAMCAN'`). The value associated with each dataset is another dictionary containing the following fields:

* **`base_dir`**
  Path to the root directory containing the subject M/EEG data.

* **`task`**
  Task label used in the recording filenames (e.g., `task-rest` in `sub-01_ses-01_task-rest_run-01_meg.fif`). Recording filenames should follow the BIDS naming convention.

* **`ending`**
  Filename suffix (including the extension) of the M/EEG recordings, such as `meg.fif`, `meg.ds`, or `4-Restin/4D`.

* **`line_freq`**
  Power-line frequency (typically 50 or 60 Hz) used during acquisition.

* **`empty_room_path`**
  Path to the directory containing empty-room recordings. The directory structure should mirror `base_dir`. If empty-room recordings are unavailable, set this field to `None`.

* **`empty_room_task`**
  Task label used for empty-room recordings (e.g., `empty_room`). Set to `None` if empty-room recordings are unavailable.

* **`empty_room_ending`**
  Filename suffix of the empty-room recordings. Set to `None` if empty-room recordings are unavailable.

* **`surfaces_dir`**
  Path to the FreeSurfer subjects directory. This directory should directly contain the FreeSurfer outputs for each subject (e.g., `sub-01`, `sub-02`, ...). If you are not using subject-specific source localization (including when using template MRI), set this field to `None`.

* **`event_file_path`**
  Path to the event files. If the event files are stored alongside the M/EEG recordings, this can be the same as `base_dir`. For CTF datasets, where event information is stored within the `.ds` directory, this can also be the same as `base_dir`. If no event files are available, set this field to `None`.

* **`event_file_task`**
  Task label used in the event filenames. If the events are embedded in the M/EEG recordings (e.g., CTF datasets), this can be the same as `task`. Set to `None` if no event files are available.

* **`event_file_ending`**
  Filename suffix of the event files. If events are embedded in the M/EEG recordings, this can be the same as `ending`. Set to `None` if no event files are available.

* **`event_of_interest`**
  Event ID corresponding to the resting-state segment. If no event file is available or the recording contains only resting-state data, set this field to `None`.

This configuration allows MEGaNorm to support data acquired from different MEG systems while maintaining a consistent interface.

### Other Required Parameters

* **`project_dir`**
  Directory where all output files and intermediate results will be stored.

  **Note:** Use the same `project_dir` across all notebooks to ensure that previously generated outputs are correctly located.

* **`conda_environment_name`**
  Name of the Conda environment used to execute the processing jobs. This environment must have **MEGaNorm** installed.

* **`slurm_username`**
  Your SLURM username (only required when running jobs on a SLURM cluster). Providing this allows MEGaNorm to monitor submitted jobs and automatically resubmit any failed or incomplete tasks.

* **`freesurfer_template_path`**
  Path to the FreeSurfer template subjects (e.g., `fsaverage` or age-specific templates) used for template-based source localization. If you do not wish to use template MRI for source localization, set this field to `None`.

* **`freesurfer_home`**
  Path to your FreeSurfer installation (i.e., the `FREESURFER_HOME` directory). This is required only when performing source localization. Otherwise, set this field to `None`.

* **`freesurfer_license`**
  Path to your FreeSurfer license file (`license.txt`). This is required only when performing source localization. Otherwise, set this field to `None`.


In [1]:
# Fill this dictionary with the appropriate information. If you have more than two dataset, you can simply add another one!
datasets = {
    'dataset_name_1': {
        # ---------------
        'base_dir': ... ,
        'task': ... ,
        "ending" : ... ,
        # ---------------
        "line_freq": ... ,
        # ---------------
        "empty_room_path" : ... ,
        "empty_room_task" : ... ,
        "empty_room_ending" : ... ,
        # ---------------
        "surfaces_dir" : ... ,
        # ---------------
        "event_file_path": ... ,
        "event_file_task": ... ,
        "event_file_ending": ... ,
        "event_of_interest": ... ,
        },
    'dataset_name_2': {
        # ---------------
        'base_dir': ... ,
        'task': ... ,
        "ending" : ... ,
        # ---------------
        "line_freq": ... ,
        # ---------------
        "empty_room_path" : ... ,
        "empty_room_task" : ... ,
        "empty_room_ending" : ... ,
        # ---------------
        "surfaces_dir" : ... ,
        # ---------------
        "event_file_path": ... ,
        "event_file_task": ... ,
        "event_file_ending": ... ,
        "event_of_interest": ... ,
        },
}

project_dir = ...
conda_environment_name = ...
slurm_username = ...

freesurfer_template_path = ...
freesurfer_home = ...
freesurfer_license = ...

# Import packages

In [10]:
import meganorm

import os
import sys
import json
import tqdm
import pickle
import warnings
import tempfile
import multiprocessing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Ignore warnings ===
warnings.filterwarnings("ignore")

# Feature Extraction Configuration
As mentioned earlier, the package applies several processing algorithms in sequence. You may wish to disable some of them or adjust their default settings to suit your data or analysis goals. This is done by passing parameters to the Config object.

In [6]:
conf = meganorm.utils.IO.Config(

    # ── Session ───────────────────────────────────────────────────────────────
    # Index of the MEG session to process (0 = first session)
    which_meg_session=0,

    # ── Sensor Selection ──────────────────────────────────────────────────────
    # Spatial grouping of sensors for analysis: "all" = no grouping, "lobe" = lobe-based, None = ungrouped
    which_layout="all",
    # Sensor modality to process: "mag", "grad", "meg", "eeg", or "opm"
    which_sensor="meg",

    # ── Channel Quality ───────────────────────────────────────────────────────
    # Automatically detect and remove channels that are noisy or flatlined
    drop_noisy_flat_channel=True,

    # ── ICA ───────────────────────────────────────────────────────────────────
    # Use elbow detection to automatically choose the number of ICA components
    apply_ica_elbow_detection=False,
    # Number of ICA components to decompose into; None = auto-determined from rank
    ica_n_component=None,
    # Maximum number of iterations for the ICA algorithm to converge
    ica_max_iter=800,
    # ICA algorithm: "fastica" (fast, common), "infomax" (information maximization), "picard" (robust)
    ica_method="fastica",
    # Apply ICA-based artifact correction (e.g., heartbeat, eye blinks)
    apply_ica=True,
    # Correlation threshold for automatically flagging ICA components as artifacts (0–1)
    auto_ica_corr_thr=0.5,
    # Whether to exclude annotated bad segments when fitting ICA
    ica_if_reject_by_annotation=True,

    # ── Filtering ─────────────────────────────────────────────────────────────
    # High-pass cutoff frequency (Hz) for bandpass filtering
    cutoffFreqLow=1.0,
    # Low-pass cutoff frequency (Hz) for bandpass filtering
    cutoffFreqHigh=80,
    # Target sampling rate (Hz) after downsampling
    resampling_rate=1000,
    # Apply a digital bandpass filter to the raw signal
    digital_filter=True,
    # Apply a notch filter to suppress power line noise (e.g., 50/60 Hz)
    notch_filter=True,

    # ── Noise Suppression ─────────────────────────────────────────────────────
    # Apply Oversampled Temporal Projection to suppress correlated sensor noise
    apply_oversampled_temporal_projection=True,
    # Apply continuous HPI (head position indicator) signal filtering
    apply_chpi_filter=False,

    # ── Head Movement ─────────────────────────────────────────────────────────
    # Correct for head movement using HPI coil tracking
    apply_Head_movement_correction=True,
    # Maximum allowed head displacement from mean position (m); segments exceeding this are flagged
    Head_movement_limit_from_mean=0.0015,

    # ── GEDAI (Generalized Eigendecomposition Artifact Identification) ────────
    # Apply GEDAI pipeline for suppressing non-biological artifacts
    apply_gedai=True,
    # GEDAI mode: "spectral" (wavelet-based), "broadband" (broadband noise), "both" (combined)
    gedai_method="both",
    # Optimization strategy for sensor selection in GEDAI: "optimize" or "gridsearch"
    sensai_method="optimize",
    # Duration (s) of data segments used in GEDAI processing
    gedai_duration=12,
    # Fractional overlap between GEDAI segments (0–1)
    gedai_overlap=0.5,
    # Noise multiplier for the preliminary broadband noise threshold in "both" mode
    gedai_preliminary_broadband_noise_multiplier=6.0,
    # Noise multiplier used for the final artifact rejection threshold
    gedai_noise_multiplier=3.0,
    # Wavelet family used for decomposition in spectral GEDAI (e.g., "haar", "db4")
    gedai_wavelet_type="haar",
    # Wavelet decomposition level: "auto" = data-driven, 0 = broadband only, int = fixed level
    gedai_wavelet_level="auto",
    # Low-pass cutoff (Hz) applied before wavelet decomposition; None = no cutoff
    gedai_wavelet_low_cutoff=None,
    # Number of oscillation cycles per epoch used in GEDAI spectral analysis
    gedai_epoch_size_in_cycles=12,
    # High-pass cutoff (Hz) applied within GEDAI to remove slow drifts
    gedai_highpass_cutoff=0.1,

    # ── Muscle Artifact Detection ─────────────────────────────────────────────
    # Z-score threshold for flagging segments as muscle artifact contaminated
    muscle_activity_thr=4,
    # Minimum duration (s) a clean segment must have to be retained after muscle artifact removal
    muscle_activity_min_length_good=0.1,
    # Frequency band (Hz) used to detect high-frequency muscle artifacts
    muscle_activity_filter_freq=(110, 140),

    # ── Environmental Noise Correction ────────────────────────────────────────
    # Apply environmental noise correction using reference channels or SSP
    apply_environmental_noise_correction=True,
    # CTF gradient compensation level applied to suppress environmental noise (1–3)
    ctf_gradient_comp_level=3,
    # Use Signal Space Projection computed from an empty-room recording for noise correction
    apply_environmental_noise_ssp_with_eroom=False,
    # Use ICA on reference MEG channels to identify and remove environmental noise components
    apply_environmental_noise_ica_with_ref_meg=True,
    # Threshold for selecting ICA components driven by reference MEG channels
    environmental_noise_ica_with_ref_meg_thr=2.5,
    # Whether to run ICA on sensor + reference channels together ("together") or independently ("separate")
    environmental_noise_ica_with_ref_meg_method="separate",
    # Metric used to compare ICA components against reference channels: "zscore" or "correlation"
    environmental_noise_ica_with_ref_meg_measure="zscore",

    # ── EEG Re-referencing ────────────────────────────────────────────────────
    # Re-referencing scheme for EEG: "average" (common average), "REST" (reference electrode standardization), "None"
    rereference_method="average",

    # ── Bad Segment / Channel Rejection ──────────────────────────────────────
    # Method for rejecting bad data segments: "autoreject" (data-driven), "fixed_thr" (manual thresholds), None
    bad_segment_removal_method="autoreject",
    # Variance threshold (T²) above which a magnetometer channel is marked bad
    mag_var_threshold=5000e-15,
    # Variance threshold (T/m)² above which a gradiometer channel is marked bad
    grad_var_threshold=5000e-13,
    # Variance threshold (V²) above which an EEG channel is marked bad
    eeg_var_threshold=40e-6,
    # Signal amplitude threshold (T) below which a magnetometer is considered flatlined
    mag_flat_threshold=10e-15,
    # Signal amplitude threshold (T/m) below which a gradiometer is considered flatlined
    grad_flat_threshold=10e-13,
    # Signal amplitude threshold (V) below which an EEG channel is considered flatlined
    eeg_flat_threshold=40e-6,
    # Number of standard deviations beyond which a segment is rejected as a z-score outlier
    zscore_std_thresh=15,

    # ── Autoreject Parameters ─────────────────────────────────────────────────
    # Number of bad channels to interpolate per epoch, swept during cross-validation
    autoreject_n_interpolates=[1, 4, 8, 16, 32],
    # Consensus fraction values (0–1) swept during autoreject cross-validation
    autoreject_consensus_percs=list(np.linspace(0, 1.0, 11)),
    # Number of cross-validation folds for autoreject threshold estimation; "auto" = data-driven
    autoreject_cv="auto",
    # Optimization method for autoreject threshold search: "bayesian_optimization" or "random_search"
    autoreject_thresh_method="bayesian_optimization",

    # ── Segmentation ──────────────────────────────────────────────────────────
    # Onset of the data window to keep, in seconds from the start of the recording
    segments_tmin=20,
    # End of the data window to keep, in seconds from the end of the recording (negative = from end)
    segments_tmax=-20,
    # Length of each data segment (s) used for spectral analysis
    segments_length=10,
    # Overlap between consecutive segments (s)
    segments_overlap=2,

    # ── Source Localization ───────────────────────────────────────────────────
    # Run the full source localization pipeline
    apply_source_localization=False,
    # Use an empty-room recording to estimate the noise covariance matrix
    apply_empty_room_recording=True,
    # Run MRI quality control checks before source localization
    apply_mri_QC=False,
    # Use a standard MRI template (e.g., fsaverage) instead of a subject-specific MRI
    apply_mri_template=False,
    # Path to FreeSurfer-preprocessed template brain derivatives directory
    freesurfer_template_path=None,
    # Path to the FreeSurfer installation root directory
    freesurfer_home=None,
    # Path to the FreeSurfer license file
    freesurfer_license=None,
    # Recompute BEM surfaces from scratch using the watershed algorithm
    make_new_watershed_bem=False,
    # Use the GCA subcortical atlas during FreeSurfer reconstruction
    gcaatlas=True,
    # Type of source space: "surface" (cortical sheet) or "volumetric" (full brain volume)
    SL_source_space="volumetric",
    # Electrical conductivity (S/m) for each tissue layer in the BEM head model (1 layer for MEG, 3 for EEG)
    SL_conductivity=(0.3,),
    # Inverse method for estimating source activity; currently only "lcmv" (beamformer) is supported
    SL_inverse_operator="lcmv",
    # Source space resolution string; controls the density of the dipole grid
    source_space_spacing="ico4",
    # Integer resolution level matching source_space_spacing (must be consistent, e.g. ico4 → 4)
    source_space_spacing_number=4,
    # Number of iterations for the final coregistration optimization step
    coregisteration_final_n_iterations=20,
    # Weight given to the nasion landmark during coregistration (higher = nasion-anchored)
    coregisteration_final_nasion_weight=10.0,
    # Method for estimating the data covariance matrix used in the beamformer
    covariance_method="empirical",

    # ── Beamformer ────────────────────────────────────────────────────────────
    # Source orientation constraint: None = unconstrained, "normal" = perpendicular to cortex,
    # "max-power" = dominant power direction, "vector" = full 3D vector output
    beamformer_pick_ori="max-power",
    # Beamformer weight normalization: "unit-noise-gain" (SNR-optimal),
    # "nai" (neural activity index), "unit-noise-gain-invariant" (required for vector beamformer)
    beamformer_weight_norm="unit-noise-gain",
    # Depth weighting factor (0–1) to correct for center-of-head bias in volumetric source space
    beamforme_depth=0.08,
    # Regularization factor (0–1) added to the data covariance matrix for numerical stability
    inverse_regularization_value=0.05,
    # Morph source estimates to a common brain template for group-level comparison
    apply_morphing=False,

    # ── Parcellation ──────────────────────────────────────────────────────────
    # Predefined atlas for parcellating source space into brain regions:
    # "aparc.a2009s" (Destrieux), "parac" (custom), or None (use annot file below)
    parcellation_parc="aparc.a2009s",
    # Path to a custom .annot parcellation file; used when parcellation_parc is None
    parcellation_annot_fname=None,

    # ── PSD Estimation ────────────────────────────────────────────────────────
    # PSD estimation method: "welch" (standard FFT-based) or "multitaper" (spectrally smooth)
    psd_method="welch",
    # Number of overlapping samples between FFT windows in Welch's method
    psd_n_overlap=1,
    # FFT length (samples) used in PSD computation
    psd_n_fft=2,
    # Length of each segment (samples) for Welch's method
    psd_n_per_seg=2,
    # Save computed PSD arrays to disk for later inspection
    save_psds=False,

    # ── Spectral Parameterization ─────────────────────────────────────────────
    # Method for decomposing PSDs into periodic and aperiodic components:
    # "fooof" (fitting oscillations & one over f) or "irasa" (irregular resampling auto-spectral analysis)
    parametrization_method="irasa",
    # IRASA resampling factors: (min_h, max_h, step) — must avoid integer ratios to suppress harmonics
    irasa_hset=(1.05, 2.0, 0.05),

    # ── FOOOF Parameters ──────────────────────────────────────────────────────
    # Lower frequency bound (Hz) of the range used to fit the FOOOF model
    fooof_freq_range_low=3,
    # Upper frequency bound (Hz) of the range used to fit the FOOOF model
    fooof_freq_range_high=40,
    # Shape of the aperiodic component: "fixed" (straight line in log-log) or "knee" (bent curve)
    aperiodic_mode="knee",
    # Min and max allowed peak width (Hz) when fitting Gaussian peaks [min, max]
    fooof_peak_width_limits=[1.0, 12.0],
    # Minimum peak height (above the aperiodic fit) required to retain a peak
    fooof_min_peak_height=0,
    # Minimum peak prominence in standard deviations above the residual for peak detection
    fooof_peak_threshold=2,
    # Directory path to save FOOOF model results; None = do not save
    fooof_res_save_path=None,

    # ── Source Epoch Saving ───────────────────────────────────────────────────
    # Save source-localized epoch data to disk after beamforming
    save_source_localized_epochs=False,

    # ── Frequency Bands ───────────────────────────────────────────────────────
    # Canonical frequency bands (Hz) used for power extraction and band ratio computation
    freq_bands={
        "Delta": (1, 3),
        "Theta": (3, 8),
        "Alpha": (8, 13),
        "Beta": (13, 30),
        "Gamma": (30, 40),
        "Broadband": (1, 40),
    },
    # Per-band offsets (Hz) applied around the individually estimated peak frequency
    # to define subject-specific band boundaries: (lower_offset, upper_offset)
    individualized_band_ranges={
        "Theta": (-2, 3),
        "Alpha": (-2, 3),
        "Beta": (-8, 9),
        "Gamma": (-5, 5),
    },


    # ── Feature Extraction ────────────────────────────────────────────────────
    # Minimum R² of the FOOOF/IRASA model fit required to include a channel's features
    min_r_squared=0.9,
    # Feature flags: True = extract and include in output, False = skip
    feature_categories={
        "Offset": True,                                      # Aperiodic intercept (log-power at 0 Hz extrapolation)
        "Exponent": True,                                    # Aperiodic slope (1/f exponent)
        "Peak_Center": False,                                # Center frequency (Hz) of each fitted oscillatory peak
        "Peak_Power": False,                                 # Power of each fitted oscillatory peak
        "Peak_Width": False,                                 # Bandwidth (Hz) of each fitted oscillatory peak
        "Adjusted_Canonical_Relative_Power": True,           # Band power / total power after aperiodic removal, fixed bands
        "Adjusted_Canonical_Absolute_Power": True,           # Absolute band power after aperiodic removal, fixed bands
        "Adjusted_Individualized_Relative_Power": False,     # Band power / total power after aperiodic removal, subject bands
        "Adjusted_Individualized_Absolute_Power": False,     # Absolute band power after aperiodic removal, subject bands
        "OriginalPSD_Canonical_Relative_Power": True,        # Band power / total power from raw PSD, fixed bands
        "OriginalPSD_Canonical_Absolute_Power": True,        # Absolute band power from raw PSD, fixed bands
        "OriginalPSD_Individualized_Relative_Power": False,  # Band power / total power from raw PSD, subject bands
        "OriginalPSD_Individualized_Absolute_Power": False,  # Absolute band power from raw PSD, subject bands
        "Adjusted_Band_Ratio": True,                         # Band power ratios after aperiodic removal
        "OriginalPSD_Band_Ratio": True,                      # Band power ratios from raw PSD
        "Hemispheric_Asymmetry_index": True,                 # (Left − Right) / (Left + Right) power per band
    },

    # ── Reproducibility ───────────────────────────────────────────────────────
    # Random seed for all stochastic steps (ICA, autoreject, etc.)
    random_state=42,
)

# Converting Participant Demographics

Demographic variable names and formats can vary substantially across datasets. For example, biological sex may be recorded as `male` in one dataset, `M` in another, or `MALE` in a third. To ensure compatibility with MEGaNorm, all demographic information must follow a consistent format across datasets.

Therefore, each dataset must contain a standardized tab-separated demographic file named:

```text
participants_bids.tsv
```

This file must be located in the root directory of the dataset (`base_dir` in `datasets`) and must contain the following columns:

* `participant_id`: Unique participant identifier
* `age`: Participant age in **years**
* `sex`: Biological sex at birth
* `eyes`: Eye condition during recording
* `diagnosis`: Participant diagnosis (e.g., control)

To create this standardized file, we provide the `make_demo_file_bids()` function. This function reads an existing demographic file, converts selected variables into a common format, and saves a BIDS-compatible TSV file ready for use in the pipeline.

The function must be applied separately to each dataset. It requires the following inputs:

* `file_dir` — Path to the input demographic file. Supported formats include CSV, TSV, and XLSX.
* `save_dir` — Directory where the standardized BIDS-compatible TSV file will be saved. This must correspond to the dataset `base_dir`.
* `id_col` — Zero-based column index containing participant IDs.
* `age_col` — Zero-based column index containing participant ages.
* `columns` — Additional demographic column specifications used to standardize variables such as sex, eye condition, and diagnosis.

The `columns` argument accepts one or more dictionaries. Each dictionary defines how a demographic variable should be generated in the output file. Two approaches are supported depending on whether the variable already exists in the input file or should be assigned a fixed value.

---

## Strategy 1: Map Existing Values to a Standard Format

Use this strategy when the demographic variable is present in the input file but uses inconsistent labels.

Each dictionary should contain:

* `col_name`: Standardized output column name
* `col_id`: Zero-based index of the source column
* `mapping`: Dictionary mapping original values to standardized values
* `single_value`: Set to `None`

Example:

```python
{
    "col_name": "sex",
    "col_id": 2,
    "mapping": {
        "M": "Male",
        "F": "Female"
    },
    "single_value": None
}
```

This converts the original values into the standardized representation before writing the output TSV file.

---

## Strategy 2: Assign a Fixed Value to All Participants

Use this strategy when all participants in a dataset share the same value for a variable. In this case, the variable does not need to exist in the original demographic file.

Each dictionary should contain:

* `single_value`: Value assigned to every participant
* `col_id`: `None`
* `mapping`: `None`

Example:

```python
{
    "col_name": "eyes",
    "col_id": None,
    "mapping": None,
    "single_value": "eyes_closed"
}
```

This creates the column and fills it with the provided value for every participant.

---

## Notes

* All column indices are **zero-based** (the first column has index `0`).
* `single_value` and `mapping` are mutually exclusive; only one conversion strategy can be used for each variable.
* Duplicate participant IDs are automatically removed, keeping the first occurrence.
* Missing diagnosis values are replaced with `"nan"`.
* The output file is always saved as a tab-separated TSV file, regardless of the input format.
* The `save_dir` argument must be the same directory specified as `base_dir` in the `datasets` configuration.


In [ ]:
# Dataset 1 - Example
meganorm.utils.IO.make_demo_file_bids(
    "path/to/your/demographicFile",
    "path/to/base_dir",
    0,
    1,
    {
        "col_name": "sex",       
        "col_id": 2,
        "mapping": {"M": "Male", "F": "Female"}, 
        "single_value": None
    }, 
    {
        "col_name": "eyes",
        "col_id": None, 
        "mapping": None,
        "single_value": "eyes_closed"
    }, 
    {
        "col_name": "diagnosis",
        "col_id": None, 
        "mapping": None,
        "single_value": "control"
    }
)

# Slurm Configuration

In [ ]:
# configs for parallel feature extraction
job_configs = {
               'module':conda_environment_name,
               'time':'02:00:00',
               'memory':'20GB',
                'partition':'normal',
                'core':3, 
                'node':1, 
                'batch_file_name':'batch_job',
                'slurm_username' : slurm_username
                }

# Parallel Feature Extraction Using Slurm

The following cell generates an `sbatch` script using `sbatch_feature_extraction_runner` and submits it to the cluster.

A principal (orchestrator) node manages the entire feature extraction workflow by:

* Setting up the required working directory
* Submitting one Slurm job per subject
* Monitoring the status of submitted jobs
* Automatically re-submitting failed jobs (up to `max_try` attempts)
* Collecting feature extraction results from all subjects
* Aggregating and saving the final output to `all_features.csv`


In [ ]:
meganorm.utils.parallel.sbatch_feature_extraction_runner(
        project_dir=project_dir,
        datasets=datasets,
        config_file=conf,
        job_configs=job_configs,
        time="80:00:00",
        mem="8GB", 
        freesurfer_home=freesurfer_home,
        freesurfer_license=freesurfer_license,
        auto_rerun=True,
        auto_collect=True,
        max_try=5)

## Submit the job to SLURM
os.chdir(project_dir)
!sbatch Features/feature_extraction_runner.sbatch